In [0]:
dbutils.widgets.text("day_folder", "retail_day_03")
dbutils.widgets.text("business_date", "2026-01-03")
day_folder = dbutils.widgets.get("day_folder")
business_date = dbutils.widgets.get("business_date")

from pyspark.sql import functions as F
from pyspark.sql.functions import to_timestamp, to_date, col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

schema = "workspace.retail_lakehouse"
landing = f"/Volumes/workspace/retail_lakehouse/landing/{day_folder}"
tracked_cols = ["city", "state"]

# --- orders: clean + incremental merge ---
orders_clean = (
    spark.read.option("header", "true").option("inferSchema", "true").csv(f"{landing}/orders.csv")
    .dropDuplicates(["order_id"])
    .dropna(subset=["order_id", "customer_id", "product_id", "amount"])
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("order_ts", to_timestamp(col("order_ts")))
    .withColumn("order_date", to_date(col("order_ts")))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("orders.csv"))
)
silver_orders_tbl = DeltaTable.forName(spark, f"{schema}.silver_orders")
(silver_orders_tbl.alias("target")
    .merge(orders_clean.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
print(f"silver_orders: {spark.table(f'{schema}.silver_orders').count()} rows")

# --- products: simple refresh (dimension rarely changes) ---
products_clean = (
    spark.read.option("header", "true").option("inferSchema", "true").csv(f"{landing}/products.csv")
    .dropDuplicates(["product_id"])
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("products.csv"))
)
(products_clean.write.format("delta").mode("overwrite").saveAsTable(f"{schema}.silver_products"))

# --- customers: SCD2 merge ---
day_customers = (
    spark.read.option("header", "true").option("inferSchema", "true").csv(f"{landing}/customers.csv")
    .dropDuplicates(["customer_id"])
    .withColumn("_scd_hash", F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in tracked_cols]), 256))
)
customers_tbl = DeltaTable.forName(spark, f"{schema}.silver_customers")
current_df = customers_tbl.toDF().filter("is_current = true")

changed = (
    day_customers.alias("s").join(current_df.alias("t"), "customer_id")
    .where("t._scd_hash != s._scd_hash").select("s.*")
    .withColumn("mergeKey", F.lit(None).cast("string"))
)
to_match = day_customers.withColumn("mergeKey", F.col("customer_id"))
staged = changed.unionByName(to_match)

(customers_tbl.alias("t")
    .merge(staged.alias("s"), "t.customer_id = s.mergeKey")
    .whenMatchedUpdate(
        condition="t.is_current = true AND t._scd_hash != s._scd_hash",
        set={"is_current": "false", "effective_to": f"CAST('{business_date}' AS DATE)"}
    )
    .whenNotMatchedInsert(values={
        "customer_id": "s.customer_id", "name": "s.name", "city": "s.city",
        "state": "s.state", "age": "s.age", "gender": "s.gender", "signup_date": "s.signup_date",
        "effective_from": f"CAST('{business_date}' AS DATE)", "effective_to": "CAST(NULL AS DATE)",
        "is_current": "true", "_scd_hash": "s._scd_hash",
        "_ingested_at": "current_timestamp()", "_source_file": "'customers.csv'"
    })
    .execute())
print(f"silver_customers: {spark.table(f'{schema}.silver_customers').count()} rows")

# --- refresh enriched (current-state join) ---
silver_orders = spark.table(f"{schema}.silver_orders")
silver_customers_current = spark.table(f"{schema}.silver_customers").filter("is_current = true")
silver_products = spark.table(f"{schema}.silver_products")

orders_enriched_df = (
    silver_orders.join(silver_customers_current, on="customer_id", how="left")
    .join(silver_products, on="product_id", how="left")
    .select(
        silver_orders["order_id"], silver_orders["customer_id"], silver_orders["product_id"],
        silver_orders["quantity"], silver_orders["amount"], silver_orders["order_ts"], silver_orders["order_date"],
        silver_customers_current["city"], silver_customers_current["state"],
        silver_customers_current["age"], silver_customers_current["gender"],
        silver_products["category"], silver_products["brand"], silver_products["price"],
    )
)
(orders_enriched_df.write.format("delta").mode("overwrite").saveAsTable(f"{schema}.silver_orders_enriched"))

# --- refresh latest-purchase window table ---
w = Window.partitionBy("customer_id").orderBy(col("order_ts").desc())
latest_purchase_df = (
    spark.table(f"{schema}.silver_orders_enriched")
    .withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")
)
(latest_purchase_df.write.format("delta").mode("overwrite").saveAsTable(f"{schema}.silver_customer_latest_purchase"))
print("silver layer fully refreshed")